In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/footfall-across-timelines/booknow_visits.csv
/kaggle/input/competitions/footfall-across-timelines/sample_submission.csv
/kaggle/input/competitions/footfall-across-timelines/booknow_theaters.csv
/kaggle/input/competitions/footfall-across-timelines/movie_theater_id_relation.csv
/kaggle/input/competitions/footfall-across-timelines/date_info.csv
/kaggle/input/competitions/footfall-across-timelines/cinePOS_booking.csv
/kaggle/input/competitions/footfall-across-timelines/cinePOS_theaters.csv
/kaggle/input/competitions/footfall-across-timelines/booknow_booking.csv
/kaggle/input/competitions/footfall-across-timelines/test.csv


In [2]:
import os
import warnings
 
import numpy as np
import pandas as pd
 
warnings.filterwarnings("ignore")
 
BASE = os.environ.get("FOOTFALL_BASE", "/kaggle/input/competitions/footfall-across-timelines/")
OUT = os.environ.get("FOOTFALL_OUT", "/kaggle/working/submission.csv")
TID, DATE, TARGET = "book_theater_id", "show_date", "audience_count"
SEED = 42
N_ORIGINS = int(os.environ.get("FOOTFALL_ORIGINS", "3"))  # rolling validation origins (1 = single window, fastest)
USE_CATBOOST = False  # True: average LightGBM with CatBoost (needs `catboost`, slower)
BUCKETS = [0, 3, 7, 14, 28, 1000]  # horizon buckets used for blend weights and interval calibration
 
 
def read(name):
    return pd.read_csv(BASE + name)
 
 
def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return np.abs(y - p).sum() / y.sum() * 100
 

In [3]:
visits = read("booknow_visits.csv")
visits[DATE] = pd.to_datetime(visits[DATE])
test = read("test.csv")
sample = read("sample_submission.csv")
print("test columns:", test.columns.tolist(), "| sample columns:", sample.columns.tolist())
 
if not {TID, DATE} <= set(test.columns):  # test may only carry an id like "<theatre>_<date>"
    try:
        parts = test[test.columns[0]].astype(str).str.rsplit("_", n=1, expand=True)
        test[TID], test[DATE] = parts[0], pd.to_datetime(parts[1])
    except Exception as e:
        raise ValueError(f"Cannot read theatre/date from test.csv columns {test.columns.tolist()}: set TID/DATE or parse manually") from e
test[DATE] = pd.to_datetime(test[DATE])
 
ORIGIN = visits[DATE].max()  # last day whose audience is known = forecast origin
test["h"] = (test[DATE] - ORIGIN).dt.days  # horizon of every test row
T = int(test["h"].max())
print(f"origin {ORIGIN.date()} | test {test[DATE].min().date()}..{test[DATE].max().date()} | horizons {test.h.min()}..{T}")
print("theatres in test:", test[TID].nunique(), "| never seen in train:", len(set(test[TID]) - set(visits[TID])))
assert test["h"].min() >= 1, "test overlaps known audience dates: check ORIGIN"
 
# Rolling validation: origin j replays a T-day forecast made j*T days before the real origin (most recent first).
ORIGINS = [ORIGIN - pd.Timedelta(days=T * j) for j in range(1, N_ORIGINS + 1)]
while len(ORIGINS) > 1 and ORIGINS[-1] - visits[DATE].min() < pd.Timedelta(days=120):
    ORIGINS.pop()  # keep enough history before the earliest origin
print("validation origins:", [o.date() for o in ORIGINS])
 

test columns: ['ID', 'book_theater_id', 'show_date'] | sample columns: ['ID', 'audience_count']
origin 2024-01-06 | test 2024-01-07..2024-02-28 | horizons 1..53
theatres in test: 817 | never seen in train: 7
validation origins: [datetime.date(2023, 11, 14), datetime.date(2023, 9, 22), datetime.date(2023, 7, 31)]


In [4]:
mapping = read("movie_theater_id_relation.csv").drop_duplicates()
 
 
def load_ledger(name):
    d = read(name)
    print(name, "| exact duplicate rows:", int(d.duplicated().sum()), "(check before dropping: could be real repeat bookings)")
    d[DATE] = pd.to_datetime(d["show_datetime"]).dt.normalize()
    d["lead"] = (d[DATE] - pd.to_datetime(d["booking_datetime"]).dt.normalize()).dt.days  # days booked ahead
    return d
 
 
bn = load_ledger("booknow_booking.csv")
cp = load_ledger("cinePOS_booking.csv").merge(mapping, on="cine_theater_id", how="inner")  # -> book_theater_id
print("cine ids mapped to >1 booknow theatre:", int((mapping.groupby("cine_theater_id")[TID].nunique() > 1).sum()))
 
for nm, d in [("booknow", bn), ("cinepos", cp)]:
    print(f"\n{nm}: shows {d[DATE].min().date()}..{d[DATE].max().date()} | latest booking {d['booking_datetime'].max()}")
    print("  lead-days quantiles:", d["lead"].quantile([0, .05, .25, .5, .75, .95, 1]).round(1).to_dict())
    print(f"  share booked same day or later: {(d['lead'] <= 0).mean():.1%}")
# Does the ledger contain bookings made AFTER the forecast origin (i.e. for the unseen test dates)?
LEDGER_EXTENDS = {}
for nm, key, d in [("booknow", "adv", bn), ("cinepos", "site", cp)]:
    post = d[pd.to_datetime(d["booking_datetime"]).dt.normalize() > ORIGIN]
    LEDGER_EXTENDS[key] = len(post) > 0
    print(f"{nm}: bookings made after origin {ORIGIN.date()}: {len(post):,} "
          f"(shows after origin: {int((d[DATE] > ORIGIN).sum()):,})")

 

booknow_booking.csv | exact duplicate rows: 1431 (check before dropping: could be real repeat bookings)
cinePOS_booking.csv | exact duplicate rows: 9926 (check before dropping: could be real repeat bookings)
cine ids mapped to >1 booknow theatre: 0

booknow: shows 2023-01-01..2024-01-06 | latest booking 2024-01-06 23:00:00
  lead-days quantiles: {0.0: 0.0, 0.05: 0.0, 0.25: 0.0, 0.5: 2.0, 0.75: 8.0, 0.95: 27.0, 1.0: 354.0}
  share booked same day or later: 30.3%

cinepos: shows 2023-01-01..2024-01-06 | latest booking 2024-01-06 18:00:00
  lead-days quantiles: {0.0: 0.0, 0.05: 0.0, 0.25: 1.0, 0.5: 4.0, 0.75: 9.0, 0.95: 24.0, 1.0: 90.0}
  share booked same day or later: 11.4%
booknow: bookings made after origin 2024-01-06: 0 (shows after origin: 0)
cinepos: bookings made after origin 2024-01-06: 0 (shows after origin: 0)


In [5]:
all_dates = pd.date_range(visits[DATE].min(), test[DATE].max(), freq="D")
theaters = sorted(set(visits[TID]) | set(test[TID]))
panel = pd.MultiIndex.from_product([theaters, all_dates], names=[TID, DATE]).to_frame(index=False)
panel = panel.merge(visits[[TID, DATE, TARGET]], on=[TID, DATE], how="left")
panel["observed"] = panel[TARGET].notna()
panel = panel.merge(test[[TID, DATE]].assign(is_test=True), on=[TID, DATE], how="left")
panel["is_test"] = panel["is_test"].fillna(False).astype(bool)
panel = panel.sort_values([TID, DATE]).reset_index(drop=True)
 
first_seen = visits.groupby(TID)[DATE].min()
panel["days_since_open"] = (panel[DATE] - panel[TID].map(first_seen)).dt.days
panel["cine_mapped"] = panel[TID].isin(cp[TID].unique()).astype(int)
print("panel", panel.shape, "| observed share:", round(panel["observed"].mean(), 3))

panel (350399, 7) | observed share: 0.505


In [6]:
di = read("date_info.csv")
print(di.head(3).to_string())
dcol = next(c for c in di.columns if "date" in c.lower())
hcol = next((c for c in di.columns if "holiday" in c.lower()), None)
if hcol is None:
    print("WARNING: no holiday column found in date_info; is_holiday set to 0")
if pd.to_datetime(di[dcol]).max() < test[DATE].max():
    print("WARNING: date_info does not cover the test window")
 
cal = pd.DataFrame({DATE: all_dates})
cal["dow"], cal["month"], cal["dom"] = cal[DATE].dt.dayofweek, cal[DATE].dt.month, cal[DATE].dt.day
cal["is_weekend"] = (cal["dow"] >= 5).astype(int)
if hcol:
    hol = pd.Series(pd.to_numeric(di[hcol], errors="coerce").values, index=pd.to_datetime(di[dcol]))
    cal["is_holiday"] = cal[DATE].map(hol[~hol.index.duplicated()]).fillna(0).clip(0, 1).astype(int)
else:
    cal["is_holiday"] = 0
hd = cal[DATE].where(cal["is_holiday"] == 1)
cal["days_since_holiday"] = (cal[DATE] - hd.ffill()).dt.days
cal["days_to_holiday"] = (hd.bfill() - cal[DATE]).dt.days  # calendars are known in advance: not leakage
 
meta = read("booknow_theaters.csv").drop_duplicates(TID)
for c in meta.columns:
    if c != TID and not pd.api.types.is_numeric_dtype(meta[c]):
        meta[c] = meta[c].astype("category").cat.codes
meta = meta.rename(columns={c: f"meta_{c}" for c in meta.columns if c != TID})
META_COLS = [c for c in meta.columns if c != TID]
 
panel = panel.merge(cal, on=DATE, how="left").merge(meta, on=TID, how="left")

    show_date day_of_week
0  2023-01-01      Sunday
1  2023-01-02      Monday
2  2023-01-03     Tuesday


In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
 
CLUSTER_FIT_END = min(ORIGINS)  # the old version fitted on all data, including validation/test periods
 
 
def fit_theater_clusters(fit_end, k_range=range(3, 9)):
    """Profile = log mean audience, volatility (CV), weekend lift (+ lat/lon columns if theatres file has them).
    k is chosen by silhouette score. Returns one row per theatre: cluster id + distance to every centroid
    (the distances act as a small learned 'embedding' of the theatre)."""
    tr = panel[panel["observed"] & (panel[DATE] <= fit_end)]
    g = tr.groupby(TID)[TARGET]
    prof = pd.DataFrame({"log_mean": np.log1p(g.mean()), "cv": g.std() / (g.mean() + 1), "n": g.count()})
    wk = tr.groupby([TID, "is_weekend"])[TARGET].mean().unstack().reindex(columns=[0, 1])
    prof["weekend_lift"] = wk[1] / (wk[0] + 1)
    geo = [c for c in META_COLS if any(t in c.lower() for t in ("lat", "lon"))]
    prof = prof.join(meta.set_index(TID)[geo])
    prof = prof[prof["n"] >= 20]
    cols = ["log_mean", "cv", "weekend_lift"] + geo
    Xs = StandardScaler().fit_transform(prof[cols].fillna(prof[cols].median()).fillna(0))
 
    ks = [k for k in k_range if k < len(prof)]
    if not ks:
        return pd.DataFrame({TID: prof.index, "cluster": 0, "cl_dist_0": 0.0})
    scores = {k: silhouette_score(Xs, KMeans(k, random_state=SEED, n_init=10).fit_predict(Xs)) for k in ks}
    best_k = max(scores, key=scores.get)
    print("silhouette by k:", {k: round(v, 3) for k, v in scores.items()}, "-> k =", best_k)
    km = KMeans(best_k, random_state=SEED, n_init=10).fit(Xs)
    out = pd.DataFrame(km.transform(Xs), index=prof.index, columns=[f"cl_dist_{i}" for i in range(best_k)])
    out.insert(0, "cluster", km.labels_)
    print("theatres per cluster:", out["cluster"].value_counts().sort_index().to_dict())
    return out.reset_index()
 
 
cl_tab = fit_theater_clusters(CLUSTER_FIT_END)
CL_COLS = [c for c in cl_tab.columns if c != TID]
panel = panel.merge(cl_tab, on=TID, how="left")
panel["cluster"] = panel["cluster"].fillna(-1).astype(int)

silhouette by k: {3: np.float64(0.272), 4: np.float64(0.298), 5: np.float64(0.321), 6: np.float64(0.328), 7: np.float64(0.303), 8: np.float64(0.277)} -> k = 6
theatres per cluster: {0: 91, 1: 223, 2: 276, 3: 23, 4: 90, 5: 67}


In [8]:
gb = panel.groupby(TID)[TARGET]
panel["_r7"] = gb.transform(lambda s: s.rolling(7, min_periods=2).mean())
panel["_r28"] = gb.transform(lambda s: s.rolling(28, min_periods=3).mean())
panel["_s28"] = gb.transform(lambda s: s.rolling(28, min_periods=3).std())
panel["_c28"] = gb.transform(lambda s: s.rolling(28, min_periods=1).count())  # open days in last 28
panel["_exp"] = gb.transform(lambda s: s.expanding(min_periods=3).mean())
GB = panel.groupby(TID)
 
# Cluster-level demand level (mean audience of the cluster per day, rolled over 28 days); shifted by h per horizon
CL = panel.groupby(["cluster", DATE])[TARGET].mean().rename("c_y").reset_index().sort_values(["cluster", DATE])
CL["_c28"] = CL.groupby("cluster")["c_y"].transform(lambda s: s.rolling(28, min_periods=3).mean())
 
 
def history_features(h):
    L = int(np.ceil(h / 7) * 7)  # smallest whole-week lag >= h, keeps the weekday pattern
    lags = pd.concat([GB[TARGET].shift(L + 7 * k).rename(f"l{k}") for k in range(8)], axis=1)
    out = pd.DataFrame(index=panel.index)
    for k in range(4):
        out[f"y_lag_{k + 1}"] = lags[f"l{k}"]
    out["dow_mean_4"] = lags.iloc[:, :4].mean(axis=1)  # same-weekday average of the last 4 known weeks
    out["dow_median_4"] = lags.iloc[:, :4].median(axis=1)
    out["dow_mean_8"] = lags.mean(axis=1)
    for src, name in [("_r7", "roll_mean_7"), ("_r28", "roll_mean_28"), ("_s28", "roll_std_28"),
                      ("_c28", "open_days_28"), ("_exp", "theatre_mean")]:
        out[name] = GB[src].shift(h)
    out["dow_vs_roll"] = out["dow_mean_4"] / (out["roll_mean_28"] + 1)
    out["tot_roll_28"] = GB["_tr28"].shift(h)  # mean ledger tickets over the last 28 observed days known at the origin
    return out

In [9]:
CAP = T + 8  # leads beyond this are lumped together; only leads up to T + gap are ever queried
 
 
def booking_cum(d, qty_col):
    d = d.assign(lead=d["lead"].clip(lower=0, upper=CAP))
    g = d.groupby([TID, DATE, "lead"])[qty_col].sum().unstack("lead", fill_value=0)
    g = g.reindex(columns=range(0, CAP + 1), fill_value=0)
    return g.iloc[:, ::-1].cumsum(axis=1).iloc[:, ::-1]  # cum[:, L] = tickets booked >= L days before the show
 
 
def booking_asof(cum, h, prefix, gap=7):
    """What you could see at the origin for a show h days away: bookings made >= h days ahead."""
    zero = pd.Series(0, index=cum.index)
    col = lambda k: cum[k] if k in cum.columns else zero
    out = pd.DataFrame({f"{prefix}_asof": col(h), f"{prefix}_old": col(h + gap)})
    out[f"{prefix}_pace"] = out[f"{prefix}_asof"] - out[f"{prefix}_old"]  # tickets added in the last `gap` days
    return out.reset_index()
 

FORCE_FULL = {"adv": None, "site": None}
FULL = {k: (LEDGER_EXTENDS[k] if FORCE_FULL[k] is None else FORCE_FULL[k]) for k in ("adv", "site")}
print("using FULL ledger totals:", FULL, "(False = only bookings made >= h days ahead)")
 
LEDGERS = [
    ("adv", booking_cum(bn, "tickets_booked"), bn[DATE].min()),
    ("site", booking_cum(cp, "tickets_sold"), cp[DATE].min()),
]
 
# Historical ledger total per theatre-day (complete for past dates) -> audience-per-ticket ratio over the last 28 days
_tot = sum(cum[0] for _, cum, _ in LEDGERS if 0 in cum.columns)
panel = panel.merge(_tot.rename("_tot").reset_index(), on=[TID, DATE], how="left")
panel["_tot"] = panel["_tot"].fillna(0)
panel.loc[panel[DATE] < min(st for _, _, st in LEDGERS), "_tot"] = np.nan
panel["_tr28"] = panel["_tot"].where(panel["observed"]).groupby(panel[TID]).transform(
    lambda s: s.rolling(28, min_periods=3).mean())
GB = panel.groupby(TID)  # rebind: history_features() reads this global
 
 
def features_at(h):
    f = pd.concat([panel, history_features(h)], axis=1)
    for prefix, cum, start in LEDGERS:
        b = booking_asof(cum, 0 if FULL[prefix] else h, prefix)
        cols = [c for c in b.columns if c not in (TID, DATE)]
        f = f.merge(b, on=[TID, DATE], how="left")
        f[cols] = f[cols].fillna(0)
        f.loc[f[DATE] < start, cols] = np.nan  # before the ledger starts = unknown, not "zero bookings"
    ch = CL[["cluster", DATE]].assign(cluster_roll_28=CL.groupby("cluster")["_c28"].shift(h))
    f = f.merge(ch, on=["cluster", DATE], how="left")
    f["theatre_vs_cluster"] = f["roll_mean_28"] / (f["cluster_roll_28"] + 1)
    f["adv_vs_dow"] = f["adv_asof"] / (f["dow_mean_4"] + 1)  # pickup relative to a normal day
    f["tot_asof"] = f[["adv_asof", "site_asof"]].sum(axis=1, min_count=1)
    f["tot_vs_dow"] = f["tot_asof"] / (f["dow_mean_4"] + 1)
    f["site_share"] = f["site_asof"] / (f["tot_asof"] + 1)
    f["cover_ratio"] = f["roll_mean_28"] / (f["tot_roll_28"] + 1)  # audience per ledger ticket lately
    f["implied_aud"] = f["tot_asof"] * f["cover_ratio"]  # ledger tickets x usual audience-per-ticket
    # Special-event signals: a movie release or surge shows up as bookings running far above normal across MANY
    # theatres on the same date (known at the origin, so no leakage), plus the calendar around holidays.
    f["mkt_surge"] = f.groupby(DATE)["adv_vs_dow"].transform("median")
    f["mkt_surge_tot"] = f.groupby(DATE)["tot_vs_dow"].transform("median")
    f["near_holiday"] = ((f["days_to_holiday"] <= 2) | (f["days_since_holiday"] <= 1)).astype(int)
    f["h"] = h
    return f
 
 
STATIC = ["dow", "month", "dom", "is_weekend", "is_holiday", "days_since_holiday", "days_to_holiday",
          "days_since_open", "cine_mapped"] + META_COLS + CL_COLS
HIST = ["y_lag_1", "y_lag_2", "y_lag_3", "y_lag_4", "dow_mean_4", "dow_median_4", "dow_mean_8",
        "roll_mean_7", "roll_mean_28", "roll_std_28", "open_days_28", "theatre_mean", "dow_vs_roll", "tot_roll_28",
        "cluster_roll_28", "theatre_vs_cluster"]
BOOK = ["adv_asof", "adv_old", "adv_pace", "site_asof", "site_old", "site_pace", "adv_vs_dow",
        "tot_asof", "tot_vs_dow", "site_share", "cover_ratio", "implied_aud"]
EVENT = ["mkt_surge", "mkt_surge_tot", "near_holiday"]
FEATS = STATIC + HIST + BOOK + EVENT + ["h"]

using FULL ledger totals: {'adv': False, 'site': False} (False = only bookings made >= h days ahead)


In [10]:
TRAIN_H = sorted({h for h in (1, 2, 3, 5, 7, 10, 14, 21, 28, 35, 42) if h <= T} | {T})
H_ALL = sorted(set(TRAIN_H) | set(range(1, T + 1)))
keep = [TID, DATE, TARGET] + FEATS
 
train_parts, val_parts, test_parts = [], [], []
for h in H_ALL:
    f = features_at(h)
    if h in TRAIN_H:
        train_parts.append(f.loc[f["observed"] & (f[DATE] <= ORIGIN), keep])
    for j, O in enumerate(ORIGINS):
        val_parts.append(f.loc[f["observed"] & (f[DATE] == O + pd.Timedelta(days=h)), keep].assign(origin_id=j))
    test_parts.append(f.loc[f["is_test"] & (f[DATE] == ORIGIN + pd.Timedelta(days=h)), keep])
train, val, test_feat = (pd.concat(p, ignore_index=True) for p in (train_parts, val_parts, test_parts))
 
# Event flag: holiday window, or market-wide booking surge in the top decile (threshold from pre-validation data only)
EVENT_THR = np.nanquantile(train.loc[train[DATE] <= min(ORIGINS), "mkt_surge_tot"].to_numpy(float), 0.90) \
    if train["mkt_surge_tot"].notna().any() else np.inf
EVENT_THR = np.inf if np.isnan(EVENT_THR) else EVENT_THR
 
 
def add_event_flag(df):
    df["is_event"] = ((df["is_holiday"] == 1) | (df["near_holiday"] == 1) | (df["mkt_surge_tot"] >= EVENT_THR)).astype(int)
 
 
for _df in (train, val, test_feat):
    add_event_flag(_df)
print("train", train.shape, "| val", val.shape, "| test", test_feat.shape, "| test rows expected", len(test))
print(f"event threshold (market surge) = {EVENT_THR:.2f} | event share of val rows: {val['is_event'].mean():.1%}")

train (2122332, 56) | val (107735, 57) | test (37185, 56) | test rows expected 37185
event threshold (market surge) = 0.00 | event share of val rows: 100.0%


In [11]:
import lightgbm as lgb
 
PARAMS = dict(objective="mae", learning_rate=0.05, n_estimators=600, num_leaves=63, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbosity=-1)
 
 
def fit_point_models(rows):
    ms = [lgb.LGBMRegressor(**PARAMS).fit(rows[FEATS], rows[TARGET])]
    if USE_CATBOOST:
        from catboost import CatBoostRegressor
        ms.append(CatBoostRegressor(loss_function="MAE", iterations=600, depth=8, learning_rate=0.08,
                                    random_seed=SEED, verbose=0).fit(rows[FEATS], rows[TARGET]))
    return ms
 
 
def predict_point(models, X):
    return np.mean([m.predict(X) for m in models], axis=0).clip(min=0)
 
 
parts, models_by_origin = [], []
for j, O in enumerate(ORIGINS):
    ms = fit_point_models(train[train[DATE] <= O])
    models_by_origin.append(ms)
    v = val[val["origin_id"] == j].copy()
    v["pred"] = predict_point(ms, v[FEATS])
    parts.append(v)
val = pd.concat(parts, ignore_index=True)
val["baseline"] = val["dow_mean_4"].fillna(val["roll_mean_28"]).fillna(0)
 
print("\nWAPE by validation origin (most recent first)")
print(val.groupby("origin_id").apply(lambda g: pd.Series({
    "origin": ORIGINS[int(g.name)].date(), "rows": len(g),
    "baseline": wape(g[TARGET], g["baseline"]), "lgbm": wape(g[TARGET], g["pred"])})).round(2))
print(f"\nOverall WAPE  baseline {wape(val[TARGET], val['baseline']):.2f}% | lgbm {wape(val[TARGET], val['pred']):.2f}%")
for e, name in [(0, "normal days"), (1, "event days")]:
    g = val[val["is_event"] == e]
    if len(g):
        print(f"  {name:12s} rows {len(g):6d} | baseline {wape(g[TARGET], g['baseline']):.2f}% | lgbm {wape(g[TARGET], g['pred']):.2f}%")
imp = pd.Series(models_by_origin[0][0].feature_importances_, index=FEATS).sort_values(ascending=False)
print("\ntop features:", imp.head(12).to_dict())


WAPE by validation origin (most recent first)
               origin   rows  baseline   lgbm
origin_id                                    
0          2023-11-14  35051     40.01  37.96
1          2023-09-22  36833     38.67  36.52
2          2023-07-31  35851     39.57  36.89

Overall WAPE  baseline 39.42% | lgbm 37.14%
  event days   rows 107735 | baseline 39.42% | lgbm 37.14%

top features: {'days_since_open': 4156, 'cl_dist_2': 2486, 'cl_dist_1': 2368, 'dom': 2200, 'cl_dist_4': 2133, 'cl_dist_5': 2114, 'dow_mean_8': 1944, 'dow': 1907, 'month': 1716, 'cl_dist_0': 1500, 'theatre_mean': 1460, 'cl_dist_3': 1260}


In [12]:
import logging
 
from joblib import Parallel, delayed
 
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
 
PROPHET_COUNTRY = None       # e.g. "IN" adds built-in national holidays on top of date_info (needs the `holidays` package)
PROPHET_MAX_THEATERS = None  # e.g. 30 for a quick smoke test; None = all theatres
N_JOBS = int(os.environ.get("FOOTFALL_JOBS", "-1"))
 
hol_dates = cal.loc[cal["is_holiday"] == 1, DATE]
HOLIDAYS = (pd.DataFrame({"holiday": "date_info_holiday", "ds": hol_dates, "lower_window": -1, "upper_window": 1})
            if len(hol_dates) else None)
 
 
def prophet_one(tid, hist, fut, hol, country):
    from prophet import Prophet
 
    out = pd.DataFrame({TID: tid, DATE: fut, "prophet": np.nan})
    hist = hist.dropna(subset=[TARGET]).sort_values(DATE)  # observed days only
    if len(hist) < 28:
        return out  # too little history: the blend falls back to LightGBM
    span = (hist[DATE].max() - hist[DATE].min()).days
    m = Prophet(
        weekly_seasonality=True, daily_seasonality=False,
        yearly_seasonality=span >= 2 * 365,  # one year cannot separate a yearly cycle from holidays
        seasonality_mode="multiplicative", changepoint_prior_scale=0.05,
        holidays=hol, uncertainty_samples=0,  # no interval simulation: much faster, we only need yhat
    )
    if country:
        m.add_country_holidays(country_name=country)
    m.fit(hist[[DATE, TARGET]].rename(columns={DATE: "ds", TARGET: "y"}))
    out["prophet"] = m.predict(pd.DataFrame({"ds": fut}))["yhat"].clip(lower=0).to_numpy()
    return out
 
 
def prophet_block(last_known, last_date, ids):
    """Fit on observed days <= last_known, forecast last_known+1 .. last_date, for the given theatres."""
    ids = sorted(ids)[:PROPHET_MAX_THEATERS]
    fut = pd.date_range(last_known + pd.Timedelta(days=1), last_date, freq="D")
    hist = panel.loc[(panel[DATE] <= last_known) & panel[TID].isin(ids), [TID, DATE, TARGET]]
    jobs = [delayed(prophet_one)(tid, g, fut, HOLIDAYS, PROPHET_COUNTRY) for tid, g in hist.groupby(TID)]
    return pd.concat(Parallel(n_jobs=N_JOBS)(jobs), ignore_index=True)
 
 
pf_val = pd.concat([
    prophet_block(O, O + pd.Timedelta(days=T), val.loc[val["origin_id"] == j, TID].unique())
    for j, O in enumerate(ORIGINS)], ignore_index=True)
val = val.drop(columns="prophet", errors="ignore").merge(pf_val, on=[TID, DATE], how="left")
print("val rows with a Prophet forecast:", f"{val['prophet'].notna().mean():.1%}")
 
# %% Step 10 - blend weights per horizon bucket, tuned on ALL validation origins pooled
GRID = np.round(np.linspace(0, 1, 11), 1)
MIN_ROWS = 200  # buckets smaller than this use the overall weight
 
 
def best_weight(g):
    """w in [0, 1]: share of LightGBM in w*lgbm + (1-w)*prophet (Prophet NaN falls back to LightGBM)."""
    pf = g["prophet"].fillna(g["pred"])
    scores = {w: wape(g[TARGET], w * g["pred"] + (1 - w) * pf) for w in GRID}
    return min(scores, key=scores.get)
 
 
val["bucket_id"] = pd.cut(val["h"], BUCKETS, labels=False)
w_global = best_weight(val)
weights = {b: (best_weight(g) if len(g) >= MIN_ROWS else w_global) for b, g in val.groupby("bucket_id")}
val["w"] = val["bucket_id"].map(weights)
val["prophet_f"] = val["prophet"].fillna(val["pred"])
val["blend"] = val["w"] * val["pred"] + (1 - val["w"]) * val["prophet_f"]
 
report = val.groupby("bucket_id").apply(lambda g: pd.Series({
    "rows": len(g), "baseline": wape(g[TARGET], g["baseline"]), "lgbm": wape(g[TARGET], g["pred"]),
    "prophet": wape(g[TARGET], g["prophet_f"]), "blend": wape(g[TARGET], g["blend"]), "w_lgbm": g["w"].iloc[0]}))
report.index = [f"h {BUCKETS[int(i)] + 1}-{BUCKETS[int(i) + 1]}" for i in report.index]
print(report.round(2))
print("\nBlend WAPE by origin:", {int(j): round(wape(g[TARGET], g["blend"]), 2) for j, g in val.groupby("origin_id")})
print(f"Overall WAPE  baseline {wape(val[TARGET], val['baseline']):.2f}% | lgbm {wape(val[TARGET], val['pred']):.2f}% | "
      f"prophet {wape(val[TARGET], val['prophet_f']):.2f}% | blend {wape(val[TARGET], val['blend']):.2f}%")

11:13:01 - cmdstanpy - INFO - Chain [1] start processing
11:13:01 - cmdstanpy - INFO - Chain [1] start processing
11:13:01 - cmdstanpy - INFO - Chain [1] start processing
11:13:01 - cmdstanpy - INFO - Chain [1] start processing
11:13:01 - cmdstanpy - INFO - Chain [1] done processing
11:13:01 - cmdstanpy - INFO - Chain [1] done processing
11:13:01 - cmdstanpy - INFO - Chain [1] done processing
11:13:01 - cmdstanpy - INFO - Chain [1] done processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1] done processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1] done processing
11:13:03 - cmdstanpy - INFO - Chain [1] done processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1] start processing
11:13:03 - cmdstanpy - INFO - Chain [1

val rows with a Prophet forecast: 89.5%
              rows  baseline   lgbm  prophet  blend  w_lgbm
h 1-3       6263.0     37.08  34.53    35.93  34.32     0.7
h 4-7       8338.0     36.07  33.35    35.12  33.16     0.8
h 8-14     14313.0     39.69  36.61    38.31  36.39     0.8
h 15-28    28853.0     37.53  35.29    37.83  35.11     0.8
h 29-1000  49968.0     41.22  39.24    43.36  39.12     0.9

Blend WAPE by origin: {0: np.float64(37.75), 1: np.float64(36.3), 2: np.float64(36.87)}
Overall WAPE  baseline 39.42% | lgbm 37.14% | prophet 40.20% | blend 36.98%


In [13]:
ALPHA = 0.2  # 80% prediction interval: 10% / 90% quantiles
Q_PARAMS = {**PARAMS, "n_estimators": 300, "learning_rate": 0.08}
 
 
def fit_quantile_models(rows):
    return [lgb.LGBMRegressor(**{**Q_PARAMS, "objective": "quantile", "alpha": a}).fit(rows[FEATS], rows[TARGET])
            for a in (ALPHA / 2, 1 - ALPHA / 2)]
 
 
def raw_interval(qm, X):
    lo, hi = qm[0].predict(X), qm[1].predict(X)
    return np.minimum(lo, hi).clip(min=0), np.maximum(lo, hi)  # guards against quantile crossing
 
 
def conformal_offsets(cal, lo, hi, min_rows=200):
    """Conformalized quantile regression: widen (or shrink) the raw interval by the (1-ALPHA) quantile of the
    calibration misses. Calibrated separately for event vs normal days and per horizon bucket, so surges and
    holidays get their own, usually wider, intervals."""
    y = cal[TARGET].to_numpy()
    score = np.maximum(lo - y, y - hi)  # > 0 means the raw interval missed the actual value
 
    def q(sc):
        n = len(sc)
        return float(np.quantile(sc, min(1.0, np.ceil((n + 1) * (1 - ALPHA)) / n)))
 
    overall = q(score)
    ev = cal["is_event"].to_numpy()
    bucket = pd.cut(cal["h"], BUCKETS, labels=False).to_numpy()
    ev_off = {e: (q(score[ev == e]) if (ev == e).sum() >= min_rows else overall) for e in (0, 1)}
    offsets = {}
    for e in (0, 1):
        for b in range(len(BUCKETS) - 1):
            m = (ev == e) & (bucket == b)
            offsets[(e, b)] = q(score[m]) if m.sum() >= min_rows else ev_off[e]
    return offsets
 
 
def apply_offsets(df, lo, hi, offsets):
    b = pd.cut(df["h"], BUCKETS, labels=False).to_numpy()
    off = np.array([offsets.get((int(e), int(k)), 0.0) for e, k in zip(df["is_event"].to_numpy(), b)])
    return np.maximum(lo - off, 0), hi + off
 
 
def pinball(y, q, tau):
    d = np.asarray(y, float) - q
    return float(np.mean(np.maximum(tau * d, (tau - 1) * d)))
 
 
def interval_report(v, lo, hi, label):
    y = v[TARGET].to_numpy()
    inside = (y >= lo) & (y <= hi)
    ev = v["is_event"].to_numpy() == 1
    print(f"{label:22s} coverage {inside.mean():.1%} (target {1 - ALPHA:.0%}) | normal {inside[~ev].mean():.1%} | "
          f"event {inside[ev].mean() if ev.any() else float('nan'):.1%} | mean width {np.mean(hi - lo):.1f} | "
          f"pinball {pinball(y, lo, ALPHA / 2) + pinball(y, hi, 1 - ALPHA / 2):.2f}")
 
 
# Validate on the most recent origin: quantile models never see the calibration window or the validation window.
O0 = ORIGINS[0]
q_end = O0 - pd.Timedelta(days=T)
qm = fit_quantile_models(train[train[DATE] <= q_end])
cal = train[(train[DATE] > q_end) & (train[DATE] <= O0)]
offs = conformal_offsets(cal, *raw_interval(qm, cal[FEATS]))
v0 = val[val["origin_id"] == 0]
lo_raw, hi_raw = raw_interval(qm, v0[FEATS])
lo_cal, hi_cal = apply_offsets(v0, lo_raw, hi_raw, offs)
interval_report(v0, np.minimum(lo_raw, v0["blend"]), np.maximum(hi_raw, v0["blend"]), "raw quantiles")
interval_report(v0, np.minimum(lo_cal, v0["blend"]), np.maximum(hi_cal, v0["blend"]), "conformal-calibrated")

raw quantiles          coverage 71.3% (target 80%) | normal nan% | event 71.3% | mean width 41.4 | pinball 8.04
conformal-calibrated   coverage 74.9% (target 80%) | normal nan% | event 74.9% | mean width 43.8 | pinball 7.95


In [14]:
final_models = fit_point_models(train)
test_feat["pred"] = predict_point(final_models, test_feat[FEATS])
 
pf_test = prophet_block(ORIGIN, test[DATE].max(), test[TID].unique())
test_feat = test_feat.drop(columns="prophet", errors="ignore").merge(pf_test, on=[TID, DATE], how="left")
w_test = pd.cut(test_feat["h"], BUCKETS, labels=False).map(weights).fillna(w_global)
test_feat["final"] = w_test * test_feat["pred"] + (1 - w_test) * test_feat["prophet"].fillna(test_feat["pred"])
 
q_end_f = ORIGIN - pd.Timedelta(days=T)
qm_f = fit_quantile_models(train[train[DATE] <= q_end_f])
cal_f = train[(train[DATE] > q_end_f) & (train[DATE] <= ORIGIN)]
offs_f = conformal_offsets(cal_f, *raw_interval(qm_f, cal_f[FEATS]))
lo_t, hi_t = apply_offsets(test_feat, *raw_interval(qm_f, test_feat[FEATS]), offs_f)
test_feat["lower"] = np.maximum(np.minimum(lo_t, test_feat["final"].to_numpy()), 0)
test_feat["upper"] = np.maximum(hi_t, test_feat["final"].to_numpy())
 
out = test.merge(test_feat[[TID, DATE, "final", "lower", "upper"]], on=[TID, DATE], how="left")
assert out["final"].notna().all(), "some test rows got no prediction"
id_col, tgt_col = sample.columns[0], sample.columns[1]
if id_col not in out.columns:
    out[id_col] = out[TID].astype(str) + "_" + out[DATE].dt.strftime("%Y-%m-%d")
sub = sample[[id_col]].merge(out[[id_col, "final"]].rename(columns={"final": tgt_col}), on=id_col, how="left")
assert sub[tgt_col].notna().all(), "submission ids do not match sample_submission"
sub.to_csv(OUT, index=False)
intervals = out[[id_col, "final", "lower", "upper"]].rename(columns={"final": tgt_col})
intervals.to_csv(OUT.replace(".csv", "_intervals.csv"), index=False)
print("saved", OUT, sub.shape, "and", OUT.replace(".csv", "_intervals.csv"))


11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] done processing
11:18:24 - cmdstanpy - INFO - Chain [1] start processing
11:18:24 - cmdstanpy - INFO - Chain [1]

saved /kaggle/working/submission.csv (37185, 2) and /kaggle/working/submission_intervals.csv
